In [1]:
import pandas as pd
import sqlite3

## Creating a database for analyzing company revenue, searching for insights, and testing hypotheses

In [2]:
df_orders = pd.read_csv('../data/processed/01_olist_orders_dataset.csv')
df_order_items = pd.read_csv('../data/processed/02_olist_order_items_dataset.csv')
df_products = pd.read_csv('../data/processed/03_olist_products_dataset.csv')
df_customers = pd.read_csv('../data/processed/04_olist_customers_dataset.csv')
df_sellers = pd.read_csv('../data/processed/05_olist_sellers_dataset.csv')

conn = sqlite3.connect('../data/database/data_mart.db') 

df_orders.to_sql('orders', conn, if_exists='replace', index=False) 
df_order_items.to_sql('order_items', conn, if_exists='replace', index=False)
df_products.to_sql('products', conn, if_exists='replace', index=False) 
df_customers.to_sql('customers', conn, if_exists='replace', index=False)
df_sellers.to_sql('sellers',conn,if_exists='replace',index=False)

3095

In [3]:
query = """
SELECT 
    order_items.order_id,
    order_items.order_item_id,
    order_items.product_id,

    orders.customer_id,
    customers.customer_unique_id,

    orders.order_purchase_timestamp,
    orders.order_delivered_customer_date,

    order_items.price,
    order_items.freight_value,
    (order_items.price + order_items.freight_value) AS revenue,

    products.product_category_name,

    customers.customer_city,
    customers.customer_state

FROM order_items 

LEFT JOIN orders  
    ON order_items.order_id = orders.order_id
    AND orders.order_purchase_timestamp >= '2017-01-01'
    AND orders.order_purchase_timestamp <= '2018-09-01'

LEFT JOIN products  
    ON order_items.product_id = products.product_id

LEFT JOIN customers 
    ON orders.customer_id = customers.customer_id
"""

data_mart = pd.read_sql(query, conn)

#### Checking the data_mart table for data corruption

In [4]:
len(df_order_items) - len(data_mart)

0

In [5]:
data_mart.isna().sum()

order_id                            0
order_item_id                       0
product_id                          0
customer_id                       371
customer_unique_id                371
order_purchase_timestamp          371
order_delivered_customer_date    2777
price                               0
freight_value                       0
revenue                             0
product_category_name            1603
customer_city                     371
customer_state                    371
dtype: int64

In [6]:
data_mart[data_mart['order_purchase_timestamp'].isna()]['revenue'].sum()/data_mart['revenue'].sum()

np.float64(0.0036197479903188686)

#### 1603 blanks in the 'product_category_name' column occurred after joining tables (blank values ​​for certain products were multiplied when joining tables). This is not an error, so all blank values ​​will be replaced with the "Unknown" category (the share of such rows in total sales is 1,3%).

#### 370 empty rows with no data from the orders or customers tables, but with data from order_items. These rows will be removed because they are statistically few in number and do not significantly impact overall revenue (< 0,5%).

In [7]:
data_mart = data_mart.dropna(subset=['order_purchase_timestamp'])

In [8]:
data_mart['product_category_name'] = data_mart['product_category_name'].fillna('Unknown')

In [9]:
data_mart.isna().sum()

order_id                            0
order_item_id                       0
product_id                          0
customer_id                         0
customer_unique_id                  0
order_purchase_timestamp            0
order_delivered_customer_date    2406
price                               0
freight_value                       0
revenue                             0
product_category_name               0
customer_city                       0
customer_state                      0
dtype: int64

In [10]:
len(data_mart)

112279

In [11]:
data_mart['revenue'].sum()

np.float64(15786203.57)

#### Data model check

In [12]:
data_mart[['price','freight_value','revenue']].head()

,price,freight_value,revenue
0,58.90,13.29,72.19
1,239.90,19.93,259.83
2,199.00,17.87,216.87
3,12.99,12.79,25.78
4,199.90,18.14,218.04


#### The revenue value was calculated correctly ('price' + 'freight_value')

#### Creating a truncated dataset (orders_agg)

In [13]:
data_mart.to_sql('data_mart',conn,if_exists='replace',index=False)
orders_agg = data_mart.groupby('order_id').agg({
    'revenue':'sum',
    'order_purchase_timestamp':'first',
    'customer_unique_id': 'first',
    'customer_state': 'first'
}).reset_index()

#### Data model check

In [14]:
data_mart.shape

(112279, 13)

In [15]:
orders_agg.shape

(98353, 5)

In [16]:
orders_agg['revenue'].sum() - data_mart['revenue'].sum()

np.float64(0.0)

In [17]:
orders_agg.to_sql('orders_agg',conn,if_exists='replace',index=False)

98353

In [18]:
data_mart.to_csv('../data/processed/00_olist_data_mart.csv', index=False)

## Two datasets were created, combining the initial data for further analysis (data_mart for analysis at the order level of the product in the order, orders_agg for analysis at the order level)

In [19]:
conn.close()